Step 1: Upload the Excel File & Create raw_data.csv

In [ ]:
# ==============================================================================
# CELL 1 / STEP 1: UPLOAD RAW FILE & INSPECT MISSING DATA
# ==============================================================================
import pandas as pd
import numpy as np
import io
from google.colab import files

print("Please upload your raw data file (either .csv or .xlsx):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Automatically handle both CSV and Excel uploads
if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
    df_raw = pd.read_excel(io.BytesIO(uploaded[file_name]), sheet_name='Survey Responses')
    # Save a copy as raw_data.csv for project submission
    df_raw.to_csv("raw_data.csv", index=False)
    print("  ✓ Detected Excel file — loaded 'Survey Responses' sheet.")
else:
    df_raw = pd.read_csv(io.BytesIO(uploaded[file_name]))
    print("  ✓ Detected CSV file — loaded directly.")

print("\n" + "="*60)
print(f"RAW DATASET LOADED: {df_raw.shape[0]} Rows x {df_raw.shape[1]} Columns")
print("="*60)

# Audit initial missing values
missing_counts = df_raw.isnull().sum()
print("\nMissing values detected per column:")
print(missing_counts[missing_counts > 0])

Please upload your raw data file (either .csv or .xlsx):


Saving Marketing Campaign Research Database.xlsx to Marketing Campaign Research Database.xlsx
  ✓ Detected Excel file — loaded 'Survey Responses' sheet.

RAW DATASET LOADED: 80 Rows x 16 Columns

Missing values detected per column:
Series([], dtype: int64)


Step 2: Handle Missing Values, Outliers & Normalization

In [ ]:
# ==============================================================================
# CELL 2 / STEP 2: PREPROCESSING USING LISTWISE DELETION (ROW DELETION)
# ==============================================================================
df_clean = df_raw.copy()
initial_count = len(df_clean)

print("="*60)
print(f"STARTING PREPROCESSING: {initial_count} Initial Records")
print("="*60)

# ------------------------------------------------------------------------------
# 2-A. TEXT CLEANING & CONDITIONAL LOGIC PROTECTION
# ------------------------------------------------------------------------------
# 1. Standardize any typographical dashes in age_group (e.g. '1822' -> '18-22')
if 'age_group' in df_clean.columns:
    df_clean['age_group'] = df_clean['age_group'].astype(str).str.replace(r'[\u2013\u2014\ufffd]', '-', regex=True)

# 2. Protect non-purchasers from accidental deletion:
# If respondent answered 'No' to purchasing, platform is logically 'Not applicable'
if 'purchase_ad_platform' in df_clean.columns and 'purchased_after_ad_3m' in df_clean.columns:
    df_clean.loc[df_clean['purchased_after_ad_3m'] == 'No', 'purchase_ad_platform'] = 'Not applicable'

# ------------------------------------------------------------------------------
# 2-B. LISTWISE DELETION (COMPLETE CASE ANALYSIS)
# ------------------------------------------------------------------------------
# Drop rows containing any missing value
df_clean = df_clean.dropna().reset_index(drop=True)
dropped_rows = initial_count - len(df_clean)

print("\n" + "-"*60)
print(f"[Action] Listwise Deletion Applied:")
print(f"  • Dropped Rows   : {dropped_rows} incomplete records ({dropped_rows/initial_count*100:.1f}%)")
print(f"  • Retained Rows  : {len(df_clean)} complete records ({len(df_clean)/initial_count*100:.1f}%)")
print(f"  • Remaining Nulls: {df_clean.isnull().sum().sum()}")
print("-"*60)

# ------------------------------------------------------------------------------
# 2-C. OUTLIER & RANGE INTEGRITY VERIFICATION
# ------------------------------------------------------------------------------
likert_cols = ['ad_trust', 'discount_importance', 'choose_promoted_product_likelihood']
for col in likert_cols:
    if col in df_clean.columns:
        # Enforce valid whole-number survey scale boundaries [1, 5]
        df_clean[col] = df_clean[col].clip(lower=1, upper=5).astype(int)
        print(f"  ✓ {col}: Verified bounds within [{df_clean[col].min()}, {df_clean[col].max()}]")

# ------------------------------------------------------------------------------
# 2-D. REDUNDANCY CLEANING & COMPOSITE SCORING
# ------------------------------------------------------------------------------
# Drop uninformative constant column
if 'status' in df_clean.columns:
    df_clean = df_clean.drop(columns=['status'])
    print("  ✓ Dropped zero-variance column: 'status'")

# Calculate composite consumer score directly on native 1.0–5.0 scale
df_clean['ad_receptivity_score'] = df_clean[likert_cols].mean(axis=1).round(2)
print("  ✓ Created 'ad_receptivity_score' on native 1.0–5.0 scale.")

# ------------------------------------------------------------------------------
# 2-E. SAVE CLEANED DATASET
# ------------------------------------------------------------------------------
output_filename = "cleaned_data.csv"
df_clean.to_csv(output_filename, index=False)

print("\n" + "="*60)
print(f"PREPROCESSING COMPLETE! Saved to '{output_filename}'")
print(f"Final Cleaned Shape: {df_clean.shape[0]} Rows x {df_clean.shape[1]} Columns")
print("="*60)

STARTING PREPROCESSING: 80 Initial Records

------------------------------------------------------------
[Action] Listwise Deletion Applied:
  • Dropped Rows   : 0 incomplete records (0.0%)
  • Retained Rows  : 80 complete records (100.0%)
  • Remaining Nulls: 0
------------------------------------------------------------
  ✓ ad_trust: Verified bounds within [1, 5]
  ✓ discount_importance: Verified bounds within [1, 5]
  ✓ choose_promoted_product_likelihood: Verified bounds within [1, 5]
  ✓ Dropped zero-variance column: 'status'
  ✓ Created 'ad_receptivity_score' on native 1.0–5.0 scale.

PREPROCESSING COMPLETE! Saved to 'cleaned_data.csv'
Final Cleaned Shape: 80 Rows x 16 Columns


Step 3: Download Cleaned Files

In [ ]:
# ==============================================================================
# CELL 3 / STEP 3: DOWNLOAD CLEANED DATASET
# ==============================================================================
from google.colab import files

print("Downloading 'cleaned_data.csv' to your local machine...")
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>